## **CREATE DATAHUB DOMAIN**

In [1]:
#!/usr/bin/env python3
import os
import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=False)

GMS = os.getenv("DATAHUB_GMS_URL", "http://datahub-gms:8080")
TOKEN = os.getenv("DATAHUB_TOKEN", "")

HEAD = {"Content-Type": "application/json"}
if TOKEN:
    HEAD["Authorization"] = f"Bearer {TOKEN}"

# list domain yang mau kamu punya di DataHub
DOMAINS = [
    {
        "id": "sales",
        "name": "Sales",
        "description": "Sales-related datasets",
    },
    {
        "id": "crm",
        "name": "CRM",
        "description": "Customer / CRM datasets",
    },
    {
        "id": "hr",
        "name": "HR",
        "description": "Human Resources datasets",
    },
    {
        "id": "catalog",
        "name": "Catalog",
        "description": "Product / catalog datasets",
    },
    {
        "id": "finance",
        "name": "Finance",
        "description": "Finance-related datasets",
    },
    {
        "id": "geo",
        "name": "Geo",
        "description": "Geographical / office datasets",
    },
]

MUTATION = """
mutation createDomain($input: CreateDomainInput!) {
  createDomain(input: $input)
}
"""

def create_domain(dom):
    payload = {
        "query": MUTATION,
        "variables": {
            "input": {
                "id": dom["id"],
                "name": dom["name"],
                "description": dom.get("description", ""),
            }
        },
    }
    r = requests.post(f"{GMS}/api/graphql", headers=HEAD, json=payload, timeout=30)
    r.raise_for_status()
    js = r.json()
    # kalau sudah ada, DataHub biasanya balikin error "DOMAIN_ALREADY_EXISTS"
    if "errors" in js:
        msg = js["errors"][0].get("message", "")
        print(f"[SKIP] {dom['id']} -> {msg}")
    else:
        print(f"[OK]   {dom['id']} created")

def main():
    print(f"Creating {len(DOMAINS)} domains to {GMS} ...")
    for d in DOMAINS:
        try:
            create_domain(d)
        except Exception as e:
            print(f"[ERR]  {d['id']} -> {e}")

if __name__ == "__main__":
    main()

Creating 6 domains to http://datahub-gms:8080 ...
[OK]   sales created
[OK]   crm created
[OK]   hr created
[OK]   catalog created
[OK]   finance created
[OK]   geo created


Setelah buat domain, baru jalankan ingestion pipeline agar bisa add datasets dengan domain yg telah dibuat.

## **CREATE DATAHUB TAGS FOR DATASET**

In [11]:
#!/usr/bin/env python3
import os
import re
import time
import requests
from typing import Dict, List, Set
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=False)

# =========================
# CONFIG
# =========================
DATAHUB_GRAPHQL = os.getenv("DATAHUB_GRAPHQL_URL", "http://datahub-gms:8080/api/graphql")
DATAHUB_TOKEN = os.getenv("DATAHUB_TOKEN", "")
TARGET_ENUM_TYPE = "DATASET_FIELD" # Kita kunci ke ini karena sudah terbukti valid

DATASET_URNS: Dict[str, str] = {
    "customers": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)",
    "employees": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.employees,PROD)",
    "products": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.products,PROD)",
    "productlines": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.productlines,PROD)",
    "orders": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.orders,PROD)",
    "orderdetails": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.orderdetails,PROD)",
    "payments": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.payments,PROD)",
    "offices": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.offices,PROD)",
}

ALIAS_MAPPING = {
    "customers": {
        "customerNumber": {"aliases": ["client id", "account id", "customer code"], "role": "PK"},
        "customerName": {"aliases": ["client name", "company name", "purchaser"], "role": "Dimension"},
        "contactLastName": {"aliases": ["surname", "contact person last name"], "role": "Dimension"},
        "contactFirstName": {"aliases": ["given name", "contact person first name"], "role": "Dimension"},
        "phone": {"aliases": ["contact number", "mobile", "telephone"], "role": "Contact"},
        "addressLine1": {"aliases": ["street address", "primary address"], "role": "Address"},
        "addressLine2": {"aliases": ["suite", "floor", "unit"], "role": "Address"},
        "city": {"aliases": ["municipality", "town"], "role": "Geo"},
        "state": {"aliases": ["region", "province"], "role": "Geo"},
        "postalCode": {"aliases": ["zip code", "post code"], "role": "Geo"},
        "country": {"aliases": ["nation"], "role": "Geo"},
        "salesRepEmployeeNumber": {"aliases": ["account manager id", "sales rep id", "assigned sales person"], "role": "FK"},
        "creditLimit": {"aliases": ["credit ceiling", "max credit", "spending limit"], "role": "Metric"},
    },
    "employees": {
        "employeeNumber": {"aliases": ["staff id", "worker id"], "role": "PK"},
        "lastName": {"aliases": ["surname"], "role": "Dimension"},
        "firstName": {"aliases": ["given name"], "role": "Dimension"},
        "extension": {"aliases": ["internal phone", "ext"], "role": "Contact"},
        "email": {"aliases": ["work email", "contact email"], "role": "Contact"},
        "officeCode": {"aliases": ["office id", "location code"], "role": "FK"},
        "reportsTo": {"aliases": ["manager id", "supervisor id", "direct report"], "role": "FK"},
        "jobTitle": {"aliases": ["position", "designation", "role"], "role": "Dimension"},
    },
    "offices": {
        "officeCode": {"aliases": ["office id", "branch code"], "role": "PK"},
        "city": {"aliases": ["office city", "branch location"], "role": "Geo"},
        "phone": {"aliases": ["office phone"], "role": "Contact"},
        "addressLine1": {"aliases": ["street"], "role": "Address"},
        "addressLine2": {"aliases": ["suite"], "role": "Address"},
        "state": {"aliases": ["region"], "role": "Geo"},
        "country": {"aliases": ["nation"], "role": "Geo"},
        "postalCode": {"aliases": ["zip"], "role": "Geo"},
        "territory": {"aliases": ["sales region", "area"], "role": "Dimension"},
    },
    "orderdetails": {
        "orderNumber": {"aliases": ["order id", "transaction id"], "role": "FK"},
        "productCode": {"aliases": ["sku", "item code"], "role": "FK"},
        "quantityOrdered": {"aliases": ["qty", "count", "units", "volume"], "role": "Metric"},
        "priceEach": {"aliases": ["unit price", "selling price", "deal price"], "role": "Metric"},
        "orderLineNumber": {"aliases": ["line sequence", "item number"], "role": "Dimension"},
    },
    "orders": {
        "orderNumber": {"aliases": ["order id", "invoice id"], "role": "PK"},
        "orderDate": {"aliases": ["purchase date", "transaction date"], "role": "Date"},
        "requiredDate": {"aliases": ["due date", "deadline"], "role": "Date"},
        "shippedDate": {"aliases": ["delivery date", "dispatch date"], "role": "Date"},
        "status": {"aliases": ["order state", "condition"], "role": "Dimension"},
        "comments": {"aliases": ["notes", "instructions"], "role": "Text"},
        "customerNumber": {"aliases": ["buyer id", "purchaser id"], "role": "FK"},
    },
    "payments": {
        "customerNumber": {"aliases": ["payer id"], "role": "FK"},
        "checkNumber": {"aliases": ["payment reference", "transaction ref"], "role": "PK"},
        "paymentDate": {"aliases": ["transaction date"], "role": "Date"},
        "amount": {"aliases": ["payment value", "total paid", "money received"], "role": "Metric"},
    },
    "productlines": {
        "productLine": {"aliases": ["category", "product family"], "role": "PK"},
        "textDescription": {"aliases": ["description", "details"], "role": "Text"},
        "htmlDescription": {"aliases": ["html content"], "role": "Text"},
        "image": {"aliases": ["picture", "photo"], "role": "Binary"},
    },
    "products": {
        "productCode": {"aliases": ["sku", "item id"], "role": "PK"},
        "productName": {"aliases": ["item name", "model name"], "role": "Dimension"},
        "productLine": {"aliases": ["category", "family"], "role": "FK"},
        "productScale": {"aliases": ["model scale", "ratio"], "role": "Dimension"},
        "productVendor": {"aliases": ["manufacturer", "brand", "maker"], "role": "Dimension"},
        "productDescription": {"aliases": ["specs", "details"], "role": "Text"},
        "quantityInStock": {"aliases": ["inventory", "stock level", "available units"], "role": "Metric"},
        "buyPrice": {"aliases": ["cost", "cogs", "purchase price"], "role": "Metric"},
        "MSRP": {"aliases": ["retail price", "list price", "catalog price"], "role": "Metric"},
    },
}

ALIAS_TAG_PREFIX = "alias:"
ROLE_TAG_PREFIX = "role:"

# =========================
# GRAPHQL HELPERS
# =========================
def _headers():
    h = {"Content-Type": "application/json"}
    if DATAHUB_TOKEN: h["Authorization"] = f"Bearer {DATAHUB_TOKEN}"
    else: h["X-DataHub-Actor"] = "urn:li:corpuser:admin"
    return h

def gql(query: str, variables: dict):
    try:
        resp = requests.post(DATAHUB_GRAPHQL, headers=_headers(), json={"query": query, "variables": variables}, timeout=30)
        return resp.json()
    except Exception as e:
        return {"errors": [{"message": str(e)}]}

# =========================
# LOGIC
# =========================
def get_dataset_fields(urn: str) -> Set[str]:
    query = """
    query getDatasetSchema($urn: String!) {
      dataset(urn: $urn) {
        schemaMetadata { fields { fieldPath } }
      }
    }
    """
    res = gql(query, {"urn": urn})
    fields = res.get("data", {}).get("dataset", {}).get("schemaMetadata", {}).get("fields", [])
    return {f["fieldPath"] for f in fields}

def generate_urn(display_name: str) -> str:
    tag_id = re.sub(r"[^A-Za-z0-9._-]", "_", display_name.replace(":", ".").replace(" ", "_")).lower().strip("._-")
    return f"urn:li:tag:{tag_id}"

def create_tag(display_name: str):
    tag_id = re.sub(r"[^A-Za-z0-9._-]", "_", display_name.replace(":", ".").replace(" ", "_")).lower().strip("._-")
    gql("mutation c($i: CreateTagInput!){createTag(input:$i)}", {"i": {"id": tag_id, "name": display_name}})

def attach_tags(d_urn: str, field_path: str, tag_urns: List[str]):
    variables = {
        "i": {
            "resourceUrn": d_urn,
            "tagUrns": tag_urns,
            "subResource": field_path,
            "subResourceType": TARGET_ENUM_TYPE 
        }
    }
    res = gql("""mutation add($i: AddTagsInput!){addTags(input:$i)}""", variables)
    if "errors" in res:
        print(f"[FAIL] {field_path}: {res['errors'][0]['message']}")
    else:
        print(f"[OK] {field_path}")

def main():
    print("[INFO] MEMULAI ENRICHMENT 2-FASE (PROVISIONING -> LINKING)...")
    
    # ---------------------------------------------------------
    # FASE 1: PROVISIONING (Kumpulkan semua tag & buat di awal)
    # ---------------------------------------------------------
    all_unique_tags = set()
    print("\n--- FASE 1: Mengumpulkan & Membuat Tag ---")
    
    for fields in ALIAS_MAPPING.values():
        for cfg in fields.values():
            for alias in cfg.get("aliases", []):
                all_unique_tags.add(f"{ALIAS_TAG_PREFIX}{alias}")
            if "role" in cfg:
                all_unique_tags.add(f"{ROLE_TAG_PREFIX}{cfg['role']}")
    
    print(f"[INFO] Ditemukan {len(all_unique_tags)} tag unik. Membuat tag ke DataHub...")
    for i, tag_label in enumerate(all_unique_tags):
        create_tag(tag_label)
        if i % 10 == 0: print(f"   ... dibuat {i+1} tag")

    # ---------------------------------------------------------
    # FASE 2: SETTLING (Waktu Tunggu Wajib)
    # ---------------------------------------------------------
    # 
    # DataHub butuh waktu untuk event 'TagCreated' diproses oleh MAE Consumer & masuk ke Search Index
    SETTLE_TIME = 15
    print(f"\n--- FASE 2: Waiting for Indexing ({SETTLE_TIME}s) ---")
    print("Menunggu Kafka & Elasticsearch sinkronisasi agar tidak terjadi 'Urn does not exist'...")
    time.sleep(SETTLE_TIME)
    print("Selesai menunggu.")

    # ---------------------------------------------------------
    # FASE 3: LINKING (Tempelkan Tag)
    # ---------------------------------------------------------
    print("\n--- FASE 3: Menempelkan Tag ke Kolom ---")
    for d_name, mapping_fields in ALIAS_MAPPING.items():
        if d_name not in DATASET_URNS: continue
        d_urn = DATASET_URNS[d_name]
        
        # Schema Discovery
        try:
            actual_fields = get_dataset_fields(d_urn)
            if not actual_fields: continue
        except: continue

        for target_field, cfg in mapping_fields.items():
            real_field_path = next((f for f in actual_fields if f.lower() == target_field.lower()), None)
            
            if not real_field_path:
                print(f"[SKIP] {d_name}: Field '{target_field}' tidak ditemukan.")
                continue

            # Karena tag sudah dibuat di Fase 1, kita tinggal generate URN-nya
            t_urns = []
            for a in cfg.get("aliases", []):
                t_urns.append(generate_urn(f"alias:{a}"))
            if "role" in cfg:
                t_urns.append(generate_urn(f"role:{cfg['role']}"))
            
            attach_tags(d_urn, real_field_path, t_urns)

if __name__ == "__main__":
    main()

[INFO] MEMULAI ENRICHMENT 2-FASE (PROVISIONING -> LINKING)...

--- FASE 1: Mengumpulkan & Membuat Tag ---
[INFO] Ditemukan 121 tag unik. Membuat tag ke DataHub...
   ... dibuat 1 tag
   ... dibuat 11 tag
   ... dibuat 21 tag
   ... dibuat 31 tag
   ... dibuat 41 tag
   ... dibuat 51 tag
   ... dibuat 61 tag
   ... dibuat 71 tag
   ... dibuat 81 tag
   ... dibuat 91 tag
   ... dibuat 101 tag
   ... dibuat 111 tag
   ... dibuat 121 tag

--- FASE 2: Waiting for Indexing (15s) ---
Menunggu Kafka & Elasticsearch sinkronisasi agar tidak terjadi 'Urn does not exist'...
Selesai menunggu.

--- FASE 3: Menempelkan Tag ke Kolom ---
[OK] customerNumber
[OK] customerName
[OK] contactLastName
[OK] contactFirstName
[OK] phone
[OK] addressLine1
[OK] addressLine2
[OK] city
[OK] state
[OK] postalCode
[OK] country
[OK] salesRepEmployeeNumber
[OK] creditLimit
[OK] employeeNumber
[OK] lastName
[OK] firstName
[OK] extension
[OK] email
[OK] officeCode
[OK] reportsTo
[OK] jobTitle
[OK] officeCode
[OK] city
[O

In [1]:
#!/usr/bin/env python3
import os
import time
import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=False)

GMS   = os.environ.get("DATAHUB_GMS_URL", "http://datahub-gms:8080")
TOKEN = os.environ.get("DATAHUB_TOKEN")
HEAD  = {"Content-Type": "application/json"}
if TOKEN:
    HEAD["Authorization"] = f"Bearer {TOKEN}"
else:
    HEAD["X-DataHub-Actor"] = "urn:li:corpuser:admin"

# --- di output-mu ketemu pola: alias.xxx (di urn/id) dan alias:xxx (di name)
# jadi kita bikin prefix yang "bebas pemisah"
BASE_PREFIXES = ("alias", "role", "dom")

DRY_RUN  = False
PAGE_SZ  = 200

def gql(q, v):
    r = requests.post(
        f"{GMS}/api/graphql",
        headers=HEAD,
        json={"query": q, "variables": v},
        timeout=30,
    )
    r.raise_for_status()
    js = r.json()
    if "errors" in js:
        raise RuntimeError(js["errors"])
    return js["data"]

def list_all_tags():
    """Return list of (urn, id, name) for all tags."""
    out, start = [], 0
    q = """
    query($start:Int!, $count:Int!) {
      search(input:{type:TAG, query:"", start:$start, count:$count}) {
        total
        searchResults {
          entity {
            urn
            ... on Tag {
              urn
              properties { name }
            }
          }
        }
      }
    }"""
    while True:
        d = gql(q, {"start": start, "count": PAGE_SZ})
        sr = d["search"]["searchResults"]
        if not sr:
            break
        for r in sr:
            ent = r["entity"]
            urn = ent["urn"]
            # di kasusmu: urn:li:tag:alias.address_1
            tag_id = urn.split("urn:li:tag:")[-1]
            props = ent.get("properties") or {}
            name  = props.get("name")  # contoh: "alias:address 1"
            out.append((urn, tag_id, name))
        start += PAGE_SZ
        if start >= d["search"]["total"]:
            break
    return out

def hard_delete(urn):
    r = requests.post(
        f"{GMS}/entities?action=delete",
        headers=HEAD,
        json={"urn": urn, "hardDelete": True},
        timeout=30,
    )
    r.raise_for_status()
    return r.json()

def match_prefix(tag_id: str, name: str) -> bool:
    """
    Kembalikan True kalau tag ini masuk salah satu BASE_PREFIXES.
    Kita normalize:
      - lowercase
      - ganti ":" jadi "."
    lalu cek prefix.
    """
    candidates = []

    if tag_id:
        candidates.append(tag_id.lower())
        candidates.append(tag_id.lower().replace(":", "."))
    if name:
        candidates.append(name.lower())
        candidates.append(name.lower().replace(":", "."))

    for cand in candidates:
        for base in BASE_PREFIXES:
            if cand.startswith(base + "."):
                return True
    return False

def main():
    all_tags = list_all_tags()

    print("== tags ditemukan dari GMS ==")
    for urn, tid, name in all_tags:
        print(f"- {urn}  (id={tid}, name={name})")

    # filter
    targets = []
    for urn, tid, name in all_tags:
        if match_prefix(tid, name):
            targets.append((urn, tid, name))

    if not targets:
        # kasih info prefix yang sekarang dipakai
        expanded = [f"{p}. / {p}:" for p in BASE_PREFIXES]
        print("\nNo matching tags with prefixes (normalized):", ", ".join(expanded))
        return

    print(f"\nFound {len(targets)} tag(s) to remove:")
    for urn, tid, name in targets:
        print(f"  - {urn}  (name={name}, id={tid})")

    if DRY_RUN:
        print("\n[DRY RUN] Set DRY_RUN=False to actually delete.")
        return

    print("\nDeleting...")
    ok = 0
    for urn, tid, name in targets:
        try:
            hard_delete(urn)
            ok += 1
            print(f"  OK  {urn}  ({name})")
            time.sleep(0.05)
        except Exception as e:
            print(f"  FAIL {urn} -> {e}")

    print(f"\nDone. Deleted {ok}/{len(targets)} tags.")

if __name__ == "__main__":
    main()

== tags ditemukan dari GMS ==
- urn:li:tag:alias.account_manager  (id=alias.account_manager, name=alias:account manager)
- urn:li:tag:alias.account_name  (id=alias.account_name, name=alias:account name)
- urn:li:tag:alias.address_1  (id=alias.address_1, name=alias:address 1)
- urn:li:tag:alias.address_2  (id=alias.address_2, name=alias:address 2)
- urn:li:tag:alias.address_line_2  (id=alias.address_line_2, name=alias:address line 2)
- urn:li:tag:alias.alamat_1  (id=alias.alamat_1, name=alias:alamat 1)
- urn:li:tag:alias.alamat_2  (id=alias.alamat_2, name=alias:alamat 2)
- urn:li:tag:alias.alamat_jalan  (id=alias.alamat_jalan, name=alias:alamat jalan)
- urn:li:tag:alias.alamat_tambahan  (id=alias.alamat_tambahan, name=alias:alamat tambahan)
- urn:li:tag:alias.amount  (id=alias.amount, name=alias:amount)
- urn:li:tag:alias.area_penjualan  (id=alias.area_penjualan, name=alias:area penjualan)
- urn:li:tag:alias.assigned_sales  (id=alias.assigned_sales, name=alias:assigned sales)
- urn:li:t